## Outline
1. load dataset
2. inspect data
3. summarize groups
4. run z-test
5. print results

In [1]:

import os

# choose your own folder here
os.environ["KAGGLEHUB_CACHE"] = "/Users/CQ/Documents/kaggle_data"

import glob # glob helps us search for files like .csv inside folders
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

import kagglehub
from kagglehub import KaggleDatasetAdapter


## 1. Load dataset

In [2]:

def load_marketing_ab_data(dataset_slug: str):
    # download dataset folder
    dataset_path = kagglehub.dataset_download(dataset_slug)

    print("Downloaded to:", dataset_path)

    # find the csv file inside that downloaded folder
    csv_files = glob.glob(os.path.join(dataset_path, "*.csv"))
    print("CSV files found:", csv_files)

    # read the first csv file
    df = pd.read_csv(csv_files[0])

    return df

In [3]:
#df = load_marketing_ab_data(dataset_slug="faviovaz/marketing-ab-testing")
#print(df.head())

## 2. Inspect data
- inspect_data()
- Purpose - prints the first essential diagnostics
    - col names
    - first few rows
    - data types
    - missing values
- Why it is useful?
    - In real data work, you should never jump straight into hypothesis testing before checking the dataset structure. 
    - eg. verify:
        - Is the group col really names "test group"?
        - Is the outcome col really binary?
        - Are there missing rows?
        - Are values stored as integers, booleans, or strings?
        

In [4]:
def inspect_data(df: pd.DataFrame) -> None:
    """
    - Print a quick overview of the dataset.
    - Before doing statistical testing, we must understand the data.
    - This helps us verify col names, missing values, and sample rows.
    Params: df - pd.DataFrame - the dataset to inspect
    return: None
    """
    # drop unamed column
    df = df.drop(columns=["Unnamed: 0"])

    # Print a separator line to make console output easier to read
    print("\n" + "=" * 60)

    # print the first few rows
    print("\nFirst 5 rows:")
    print(df.head())

    # print data types
    print("\nData types:")
    print(df.dtypes)

    # print missing-value counts
    print("\nMissing values per column:")
    print(df.isna().sum())

    # print unique value %
    for col in ["test group", "converted", "most ads day"]:
      print(f"\nColumn: {col}")
      display((df[col].value_counts(normalize=True) * 100).round(2))
      print("-" * 40)

    # print another separator line
    print("=" * 60 + "\n")

In [5]:
#inspect=inspect_data(df)
#inspect

## 3. AB test summary
prepare_ab_summary()
- Purpose: this creates a group summary table with:
    - num of users
    - num of conversions
    - conversion rate
- Why it is useful
    - Before formal testing, you should always look at the descriptive statistics first. 
    - For each group, the conversion rate is: p = num of conversions / num of users
    - This table lets you see the practical diff before doing significance testing
- Output
    - A summary DF like: group; n; conversions; conversion rate


In [6]:
def prepare_ab_summary(
        df: pd.DataFrame, 
        group_col: str="test group", 
        outcome_col: str="converted") -> pd.DataFrame:
    """
    - Build a summary table for the AB groups. 
    - AB testing usually starts with group-level summaries. 
    - We want sample size, # of conversions, and conversion rate. 
    Params: df; group_col; outcome_ col
    Returns: pd.DataFrame - A summary table with group size, conversions, and conversion rate.
    """
    # group by the experiment group and aggregate useful statistics
    summary = (
        df.groupby(group_col)[outcome_col]
        .agg(["count", "sum", "mean"])
        .rename(columns={
            "count": "n_users",
            "sum": "n_converted", 
            "mean": "conversion_rate"
        })
        .reset_index()
    )       

    # print the summary for quick inspection
    print("A/B Summary:")
    #print(summary)

    # return the summary table
    return summary


In [7]:

#summary = prepare_ab_summary(df)
#display(summary.style.format({"conversion_rate": "{:.2%}"}))


## 4. Proportion test
run_two_proportion_test()
- Purpose 
    - this is the core statistic function
    - It compares the conversion rates of two groups using a two-proportion z-test.
- Why this test?
    - Your outcome is binary - converted=1; not converted=0
    - That means each group is essentially a binomial sample, and comparing two proportions is exactly what the two-proportion z-test is for. 
- Hypotheses: H0: Pcontro=Ptreatment; H1: Pcontrol != Ptreatment
- What the function computes
    - It extracts: control conversions| treatment conversions | control sample size | treatment sample size
    - proportions_ztest(count=counts, nobs=nobs, alternative="two-sided)
        - count: 1000 visitors; nobs = # of observations = 100 conversions
    - This returns: z-statistic; p-values
- Interpretation rule: if p-value < alpha then reject H0; otherwise, fail to reject H0. 
- Return a dict: to keep all outputs together: rates | counts | test statistic | p-value | conclusion


In [8]:
def run_two_proportion_test(
        df: pd.DataFrame,
        group_col: str="test group", 
        outcome_col: str="converted",
        control_name: str="psa", # public service Announcement (control group)
        treatment_name: str="ad", # advertising (treatment group)
        alpha:float=0.05
) -> dict:
    """
    - Run a two-proportion z-test for conversion rates. 
    - This is the main inferential step in a binary A/B test. 
    - We compare whether the conversion rates differ significantly. 

    # Statistical idea
    - Suppose: p_control & p_treatment (both in conversion % )
    - H0: p_control = p_treatment; H1: p_control != p_treatment

    # Params:
    - df; group_col ("test group"); outcome_col ("converted"); control_name (ad); treatment_name (psa); alpha (significance level, may 0.05)
    # Return: dict contains conversion rates, z-statistic, p-value, conclusion
    """
   
    control_df = df[df[group_col] == control_name]  # Select only the control rows
    treatment_df = df[df[group_col] == treatment_name] # select only the treatment rows
    control_success = control_df[outcome_col].sum() # count conversions in the control group
    treatment_success = treatment_df[outcome_col].sum() # count conversions in the treatment group
    control_n = control_df.shape[0] # count total observations in each group
    treatment_n = treatment_df.shape[0]
    counts = [control_success, treatment_success] # build the "success counts" array for the z-test
    nobs = [control_n, treatment_n] # num of observe; build the "sample sizes" array for the z-test
    z_stat, p_value = proportions_ztest(count=counts, nobs=nobs, alternative='two-sided') # run the two-sided z-test for two proportions
    
    control_rate = (control_success / control_n).round(2) # conversion rates
    treatment_rate = (treatment_success / treatment_n).round(2)

    # decide whether to reject the H0
    if p_value < alpha: conclusion = "Reject H0: the conversion rates are significantly different."
    else: conclusion = "Fail to reject H0: no statistically significant difference was detected."

    # Package results into a dict
    results = {
        "control_group": control_name,
        "treatment_group": treatment_name,
        "control_n": control_n,
        "treatment_n": treatment_n,
        "control_conversions": int(control_success),
        "treatment_conversions": int(treatment_success),
        "control_rate": control_rate,
        "treatment_rate": treatment_rate,
        "difference_in_rate": (treatment_rate - control_rate).round(2),
        "z_statistic": z_stat.round(2),
        "p_value": p_value.round(4),
        "alpha": alpha,
        "conclusion": conclusion
    }
    return results



In [9]:
#res = run_two_proportion_test(df, "test group", "converted", "psa", "ad", 0.05)
#res

## 5. Print results

In [10]:
def print_test_results(results: dict) -> None:
    """
    Print the A/B test results in a readable format.

    Why this function exists:
    - Raw dictionaries are correct but not always easy to read.
    - This function turns the statistical output into a clean report.

    Parameters
    ----------
    results : dict
        Output dictionary from run_two_proportion_test().

    Returns
    -------
    None
    """

    # print a title
    print("\nA/B Test Results")
    print("-" * 60)

    # print sample sizes
    print(f"Control group ({results['control_group']}): n = {results['control_n']}")
    print(f"Treatment group ({results['treatment_group']}): n = {results['treatment_n']}")

    # print conversion counts
    print(f"Control conversions: {results['control_conversions']}")
    print(f"Treatment conversions: {results['treatment_conversions']}")

    # print conversion rates formatted as percentages
    print(f"Control conversion rate: {results['control_rate']:.4%}")
    print(f"Treatment conversion rate: {results['treatment_rate']:.4%}")

    # print the difference
    print(f"Difference (treatment - control): {results['difference_in_rate']:.4%}")

    # print z-statistic and p-value
    print(f"Z-statistic: {results['z_statistic']:.4f}")
    print(f"P-value: {results['p_value']:.6f}")

    # print the final interpretation
    print(f"Conclusion: {results['conclusion']}")

In [11]:
#p_res = print_test_results(res)

## 6. main

In [12]:
def main() -> None:
    """
    Main driver function
    - It orchestrates the full workflow in one place. 
    - A good main() makes the script easier to read from top to bottom

    Workflow:
    1. Load data
    2. Inspect data
    3. Create summary table
    4. Run AB test
    5. Print results
    """

In [13]:
# the Kaggle dataset identifier
dataset_slug = "faviovaz/marketing-ab-testing"
# Step 1: Load data
df = load_marketing_ab_data(dataset_slug)
# Step 2: Inspect the dataset
inspect_data(df)
# Step 3: Create a group-level summary table
prepare_ab_summary(df, group_col="test group", outcome_col="converted")
# Step 4: Run the two-proportion z-test
results = run_two_proportion_test(
        df,
        group_col="test group",
        outcome_col="converted",
        control_name="psa",
        treatment_name="ad",
        alpha=0.05
    )
# Step 5: Print the final results
print_test_results(results)


Downloaded to: /Users/CQ/Documents/kaggle_data/datasets/faviovaz/marketing-ab-testing/versions/1
CSV files found: ['/Users/CQ/Documents/kaggle_data/datasets/faviovaz/marketing-ab-testing/versions/1/marketing_AB.csv']


First 5 rows:
   user id test group  converted  total ads most ads day  most ads hour
0  1069124         ad      False        130       Monday             20
1  1119715         ad      False         93      Tuesday             22
2  1144181         ad      False         21      Tuesday             18
3  1435133         ad      False        355      Tuesday             10
4  1015700         ad      False        276       Friday             14

Data types:
user id           int64
test group       object
converted          bool
total ads         int64
most ads day     object
most ads hour     int64
dtype: object

Missing values per column:
user id          0
test group       0
converted        0
total ads        0
most ads day     0
most ads hour    0
dtype: int64

Column: 

test group
ad     96.0
psa     4.0
Name: proportion, dtype: float64

----------------------------------------

Column: converted


converted
False    97.48
True      2.52
Name: proportion, dtype: float64

----------------------------------------

Column: most ads day


most ads day
Friday       15.75
Monday       14.81
Sunday       14.52
Thursday     14.11
Saturday     13.89
Wednesday    13.76
Tuesday      13.17
Name: proportion, dtype: float64

----------------------------------------

A/B Summary:

A/B Test Results
------------------------------------------------------------
Control group (psa): n = 23524
Treatment group (ad): n = 564577
Control conversions: 420
Treatment conversions: 14423
Control conversion rate: 2.0000%
Treatment conversion rate: 3.0000%
Difference (treatment - control): 1.0000%
Z-statistic: -7.3700
P-value: 0.000000
Conclusion: Reject H0: the conversion rates are significantly different.
